# Detekcja anomalii w rzeczywistych logach auth.log

Notebook analizuje rzeczywisty plik `data/raw/auth_all_real.log` i porownuje trzy podejscia do wykrywania anomalii:

- **detekcja regulowa** - proste, jawne reguly bez uczenia maszynowego,
- **Isolation Forest** - model ML wykrywajacy obserwacje odstajace,
- **Local Outlier Factor** - model ML porownujacy gestosc lokalnego sasiedztwa punktow.

Celem projektu jest sprawdzenie, czy podejscie ML daje lepsze lub gorsze wyniki niz prosta detekcja regulowa dla logow uwierzytelniania Linux.

## 1. Przygotowanie srodowiska

Ta sekcja importuje biblioteki, ustawia sciezke projektu i wczytuje funkcje przygotowane w katalogu `src`. Dzieki temu notebook korzysta z tego samego kodu co projekt, a nie z osobnych funkcji pisanych tylko w notebooku.

In [ ]:
# Path/sys pozwalaja uruchamiac notebook zarowno z glownego katalogu projektu,
# jak i bezposrednio z katalogu notebooks.
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from config import RAW_LOG_PATH
from src.auth_parser import parse_lines_grouped
from src.features import records_to_features
from src.log_loader import load_logs
from src.ml_detector import (
    predict_anomalies,
    predict_local_outlier_factor,
    train_isolation_forest,
    train_local_outlier_factor,
)
from src.preprocessing import (
    auto_label_records,
    calculate_detection_statistics,
    calculate_labeled_metrics,
    compare_detection_methods,
    confusion_matrices,
    detections_to_dataframe,
    records_to_dataframe,
)

# Ustawienia wykresow, aby wszystkie rysunki mialy spojny styl w sprawozdaniu.
plt.style.use('default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

## 2. Wczytanie i parsowanie danych

Parser `auth_parser.py` zamienia kazda linie auth.log na obiekt `LogRecord`. Z rekordu wyciagane sa m.in. czas, host, usluga/proces, uzytkownik, adres IP, port, typ zdarzenia i informacja o sukcesie lub porazce operacji.

In [ ]:
# Wczytujemy surowe linie z auth_all_real.log.
lines = load_logs(RAW_LOG_PATH)

# Kazda niepusta linia staje sie jednym rekordem LogRecord.
records = parse_lines_grouped(lines)

# df to czytelna tabela rekordow do analiz opisowych.
df = records_to_dataframe(records)

# features to tabela liczbowa uzywana przez Isolation Forest i LOF.
features = records_to_features(records)

# Etykiety sa wyznaczane automatycznie na podstawie jednoznacznie podejrzanych zdarzen.
# anomaly=1: Failed password, Invalid user, authentication failure, brute-force SSH.
# anomaly=0: poprawne logowania, zwykle sudo, sesje uzytkownikow, CRON i pozostale neutralne zdarzenia.
labels = auto_label_records(records)

print(f'Plik: {RAW_LOG_PATH}')
print(f'Liczba linii: {len(lines)}')
print(f'Liczba rekordow: {len(records)}')
print(f'Liczba cech ML: {features.shape[1]}')
df.head()

## 3. Struktura zbioru

Tutaj sprawdzamy, jakie typy zdarzen i uslugi dominuja w logach. To jest wazne, bo modele ML ucza sie struktury danych: jezeli zbior jest zdominowany przez jeden typ aktywnosci, modele moga slabiej wykrywac rzadsze, ale istotne zdarzenia.

In [ ]:
# Liczymy rozklady najwazniejszych kolumn.
event_counts = df['event_type'].value_counts()
service_counts = df['service'].value_counts()
label_counts = labels.value_counts().rename(index={0: 'normal', 1: 'anomaly'})

display(event_counts.to_frame('count').head(20))
display(service_counts.to_frame('count').head(20))
display(label_counts.to_frame('count'))

In [ ]:
# Wykres 1: najczestsze typy zdarzen.
# Pokazuje, czy log jest zdominowany przez zdarzenia SSH, CRON, sudo czy inne uslugi.
ax = event_counts.head(15).sort_values().plot(kind='barh', title='Najczestsze typy zdarzen')
ax.set_xlabel('Liczba zdarzen')
ax.set_ylabel('Typ zdarzenia')
plt.tight_layout()

In [ ]:
# Wykres 2: udzial rekordow normalnych i anomalii wedlug automatycznych etykiet.
# To pokazuje poziom niezbalansowania klas, ktory wplywa na metryki precision/recall/F1.
colors = ['#4c78a8', '#e45756']
ax = label_counts.plot(kind='bar', color=colors, title='Rozklad automatycznych etykiet')
ax.set_xlabel('Klasa')
ax.set_ylabel('Liczba rekordow')
plt.xticks(rotation=0)
plt.tight_layout()

In [ ]:
# Wykres 3: liczba zdarzen w czasie, agregacja godzinowa.
# Pozwala zobaczyc okresy wzmozonej aktywnosci i potencjalne fale atakow.
time_df = df.dropna(subset=['timestamp']).copy()
time_df['hour_bucket'] = pd.to_datetime(time_df['timestamp']).dt.floor('h')
time_df['label'] = labels.values

hourly = time_df.groupby(['hour_bucket', 'label']).size().unstack(fill_value=0).rename(columns={0: 'normal', 1: 'anomaly'})
ax = hourly.plot(kind='line', title='Aktywnosc w czasie: normalne zdarzenia vs anomalie')
ax.set_xlabel('Czas')
ax.set_ylabel('Liczba rekordow na godzine')
plt.tight_layout()

In [ ]:
# Wykres 4: heatmapa aktywnosci wedlug dnia tygodnia i godziny.
# Wysokie wartosci w nietypowych godzinach moga wskazywac automatyczne skanowanie lub brute-force.
time_df['day_name'] = pd.to_datetime(time_df['timestamp']).dt.day_name()
time_df['hour'] = pd.to_datetime(time_df['timestamp']).dt.hour
heatmap_data = time_df.pivot_table(index='day_name', columns='hour', values='event_type', aggfunc='count', fill_value=0)
heatmap_data = heatmap_data.reindex(['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'])

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(heatmap_data.values, aspect='auto', cmap='YlOrRd')
ax.set_title('Heatmapa liczby zdarzen wedlug dnia tygodnia i godziny')
ax.set_xlabel('Godzina')
ax.set_ylabel('Dzien tygodnia')
ax.set_xticks(range(24))
ax.set_yticks(range(len(heatmap_data.index)))
ax.set_yticklabels(heatmap_data.index)
fig.colorbar(im, ax=ax, label='Liczba zdarzen')
plt.tight_layout()

## 4. Najczesciej atakowane konta i adresy IP

Dla zdarzen `Failed password`, `Invalid user` i `authentication failure` sprawdzamy, ktore konta i adresy IP pojawiaja sie najczesciej. To sa przydatne wykresy do sprawozdania, bo pokazuja konkretne zrodla i cele prob logowania.

In [ ]:
# Wybieramy tylko zdarzenia nieudanych prob uwierzytelnienia.
attacks = df[df['event_type'].isin(['ssh_failed_password', 'ssh_invalid_user', 'authentication_failure'])]

# Najczesciej atakowane nazwy kont.
attacked_users = attacks['user'].dropna().value_counts().head(20)

# Najczesciej wystepujace adresy IP w podejrzanych zdarzeniach.
top_ips = attacks['ip_address'].dropna().value_counts().head(20)

display(attacked_users.to_frame('failed_events'))
display(top_ips.to_frame('failed_events'))

In [ ]:
# Wykres 5 i 6: najczesciej atakowane konta oraz najczestsze adresy IP.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
attacked_users.head(10).sort_values().plot(kind='barh', ax=axes[0], title='Najczesciej atakowane konta')
top_ips.head(10).sort_values().plot(kind='barh', ax=axes[1], title='Najczestsze adresy IP w atakach')
axes[0].set_xlabel('Liczba nieudanych zdarzen')
axes[1].set_xlabel('Liczba nieudanych zdarzen')
plt.tight_layout()

In [ ]:
# Wykres 7: rozklad typow podejrzanych zdarzen SSH/PAM.
# Pokazuje, jaki rodzaj problemu dominuje: hasla, nieistniejacy uzytkownicy czy PAM.
attack_type_counts = attacks['event_type'].value_counts()
ax = attack_type_counts.plot(kind='bar', color='#e45756', title='Typy podejrzanych zdarzen uwierzytelniania')
ax.set_xlabel('Typ zdarzenia')
ax.set_ylabel('Liczba rekordow')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()

## 5. Detekcja regulowa

Detekcja regulowa dziala bez uczenia maszynowego. Reguly sa jawne i interpretowalne: wykrywaja m.in. nieudane hasla SSH, nieistniejacych uzytkownikow, bledy PAM, logowanie roota, brute-force z jednego IP, nietypowe godziny oraz uzycie sudo.

In [ ]:
# Uruchamiamy wszystkie reguly i zamieniamy wyniki na tabele.
stats = calculate_detection_statistics(records)
detections_df = detections_to_dataframe(records)

print(stats)
display(detections_df['rule'].value_counts().to_frame('detections'))
detections_df.head(20)

In [ ]:
# Wykres 8: liczba detekcji wedlug reguly.
# Reguly z najwieksza liczba trafien wskazuja dominujacy typ zagrozenia w danych.
rule_counts = detections_df['rule'].value_counts()
ax = rule_counts.sort_values().plot(kind='barh', color='#f58518', title='Detekcje regulowe wedlug typu reguly')
ax.set_xlabel('Liczba detekcji')
ax.set_ylabel('Regula')
plt.tight_layout()

## 6. Cechy dla modeli ML

Modele ML nie analizuja bezposrednio tekstu logu. Kazdy rekord jest zamieniany na zestaw cech liczbowych, np. godzina, dzien tygodnia, typ zdarzenia, dlugosc nazwy uzytkownika, obecnosc IP, port, wynik operacji, flaga sudo/cron oraz liczba nieudanych prob w oknie czasowym.

In [ ]:
# Podglad tabeli cech uzywanej przez Isolation Forest i LOF.
features.head()

In [ ]:
# Wykres 9: histogram liczby nieudanych prob w oknie czasowym.
# Ta cecha pomaga modelom i regulom identyfikowac brute-force SSH.
ax = features['failed_login_count_window'].plot(kind='hist', bins=30, color='#72b7b2', title='Rozklad liczby nieudanych prob w oknie czasowym')
ax.set_xlabel('Liczba nieudanych prob w oknie')
plt.tight_layout()

## 7. Modele ML: Isolation Forest i Local Outlier Factor

Oba modele traktuja anomalie jako punkty odstajace w przestrzeni cech. W przeciwienstwie do regul, modele nie wiedza jawnie, czym jest `Failed password` lub `Invalid user`; widza tylko liczby opisujace rekord.

In [ ]:
# Trenujemy Isolation Forest i wykonujemy predykcje.
isolation_model = train_isolation_forest(features)
isolation_predictions = predict_anomalies(isolation_model, features)

# Trenujemy Local Outlier Factor i wykonujemy predykcje.
lof_model = train_local_outlier_factor(features)
lof_predictions = predict_local_outlier_factor(lof_model, features)

# Porownujemy liczbe wykrytych anomalii przez kazda metode.
comparison_df = compare_detection_methods(records, isolation_predictions, lof_predictions)
display(comparison_df)

In [ ]:
# Wykres 10: ile anomalii wykryla kazda metoda.
ax = comparison_df.set_index('method')['detected_anomalies'].plot(kind='bar', color=['#f58518', '#54a24b', '#b279a2'], title='Liczba anomalii wykrytych przez metody')
ax.set_xlabel('Metoda')
ax.set_ylabel('Liczba wykrytych anomalii')
plt.xticks(rotation=0)
plt.tight_layout()

In [ ]:
# Wykres 11: pokrycie detekcji miedzy metodami.
# Pokazuje, ile wykryc ML pokrywa sie z detekcja regulowa.
rule_indices = set(detections_df['record_index'].unique())
isolation_indices = {index for index, prediction in enumerate(isolation_predictions) if prediction == -1}
lof_indices = {index for index, prediction in enumerate(lof_predictions) if prediction == -1}

overlap_df = pd.DataFrame([
    {'category': 'Rule only', 'count': len(rule_indices - isolation_indices - lof_indices)},
    {'category': 'Isolation only', 'count': len(isolation_indices - rule_indices - lof_indices)},
    {'category': 'LOF only', 'count': len(lof_indices - rule_indices - isolation_indices)},
    {'category': 'Rule + Isolation', 'count': len((rule_indices & isolation_indices) - lof_indices)},
    {'category': 'Rule + LOF', 'count': len((rule_indices & lof_indices) - isolation_indices)},
    {'category': 'Isolation + LOF', 'count': len((isolation_indices & lof_indices) - rule_indices)},
    {'category': 'All methods', 'count': len(rule_indices & isolation_indices & lof_indices)},
])

display(overlap_df)
ax = overlap_df.set_index('category')['count'].sort_values().plot(kind='barh', title='Pokrycie detekcji miedzy metodami')
ax.set_xlabel('Liczba rekordow')
plt.tight_layout()

## 8. Confusion matrix i metryki

Metryki sa liczone wzgledem automatycznych etykiet. W tym projekcie etykiety nie pochodza z recznej analizy eksperta, tylko z jasnych wzorcow w logach. Dlatego wyniki nalezy interpretowac jako porownanie metod wobec przyjetej definicji anomalii.

In [ ]:
# Liczymy Accuracy, Precision, Recall i F1 dla kazdej metody.
metrics_df = calculate_labeled_metrics(records, labels, isolation_predictions, lof_predictions)
matrices = confusion_matrices(records, labels, isolation_predictions, lof_predictions)

display(metrics_df)
for method, matrix in matrices.items():
    print(method)
    display(matrix)

In [ ]:
# Wykres 12: porownanie czterech metryk metod.
metric_plot = metrics_df.set_index('method')[['accuracy', 'precision', 'recall', 'f1']]
ax = metric_plot.plot(kind='bar', title='Porownanie metryk metod')
ax.set_ylabel('Wartosc [%]')
ax.set_ylim(0, 105)
plt.xticks(rotation=0)
plt.tight_layout()

In [ ]:
# Wykres 13: confusion matrix jako heatmapy.
# Daje szybki podglad TP/FP/TN/FN dla kazdej metody.
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (method, matrix) in zip(axes, matrices.items()):
    im = ax.imshow(matrix.values, cmap='Blues')
    ax.set_title(method)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(matrix.columns)
    ax.set_yticks([0, 1])
    ax.set_yticklabels(matrix.index)
    for row in range(matrix.shape[0]):
        for col in range(matrix.shape[1]):
            ax.text(col, row, int(matrix.values[row, col]), ha='center', va='center', color='black')
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.8)
plt.tight_layout()

## 9. Wnioski z wynikow

Ponizsza komorka generuje wnioski automatycznie na podstawie tabel wynikowych. Tekst mozna bezposrednio wykorzystac jako punkt wyjscia do sprawozdania.

In [ ]:
best_f1 = metrics_df.sort_values('f1', ascending=False).iloc[0]
best_precision = metrics_df.sort_values('precision', ascending=False).iloc[0]
best_recall = metrics_df.sort_values('recall', ascending=False).iloc[0]
rule_row = metrics_df[metrics_df['method'] == 'Rule-based'].iloc[0]
if_row = metrics_df[metrics_df['method'] == 'Isolation Forest'].iloc[0]
lof_row = metrics_df[metrics_df['method'] == 'Local Outlier Factor'].iloc[0]

print('PODSUMOWANIE ZBIORU')
print(f'- Liczba rekordow: {len(records)}')
print(f'- Liczba automatycznie etykietowanych anomalii: {int(labels.sum())}')
print(f'- Liczba rekordow normalnych: {int((labels == 0).sum())}')
print(f'- Najczestszy typ zdarzenia: {event_counts.index[0]} ({event_counts.iloc[0]})')
print(f'- Najczesciej atakowane konto: {attacked_users.index[0] if not attacked_users.empty else "brak"}')
print(f'- Najczestszy adres IP w nieudanych probach: {top_ips.index[0] if not top_ips.empty else "brak"}')

print('\nWNIOSKI O DETEKCJI REGULOWEJ')
print(f'- Detekcja regulowa uzyskala F1={rule_row.f1}%, precision={rule_row.precision}% i recall={rule_row.recall}%.')
print('- Wysoki recall oznacza, ze reguly pokrywaja prawie wszystkie zdarzenia oznaczone automatycznie jako anomalie.')
print('- Wynik jest wysoki, bo etykiety automatyczne opieraja sie na podobnych jawnych sygnaturach: Failed password, Invalid user, authentication failure i brute-force.')
print('- Reguly sa latwe do wyjasnienia w sprawozdaniu, ale moga nie wykryc nietypowych atakow, ktore nie pasuja do zapisanych wzorcow.')

print('\nWNIOSKI O MODELACH ML')
print(f'- Isolation Forest uzyskal F1={if_row.f1}%, precision={if_row.precision}% i recall={if_row.recall}%.')
print(f'- Local Outlier Factor uzyskal F1={lof_row.f1}%, precision={lof_row.precision}% i recall={lof_row.recall}%.')
print('- Modele ML wykrywaja punkty odstajace w przestrzeni cech, ale nie znaja semantyki komunikatow SSH/PAM.')
print('- Nizszy recall modeli ML oznacza, ze wiele zdarzen typu Failed password lub Invalid user nie jest dla nich wystarczajaco odstajace liczbowo.')
print('- LOF moze lepiej wychwytywac lokalnie nietypowe punkty, a Isolation Forest lepiej izoluje globalne odstepstwa, ale oba podejscia sa wrazliwe na dobor cech i parametr contamination.')

print('\nPOROWNANIE METOD')
print(f'- Najwyzszy F1: {best_f1.method} ({best_f1.f1}%).')
print(f'- Najwyzsza precision: {best_precision.method} ({best_precision.precision}%).')
print(f'- Najwyzszy recall: {best_recall.method} ({best_recall.recall}%).')
print('- W tym zbiorze prosta detekcja regulowa jest skuteczniejsza dla jednoznacznych zdarzen uwierzytelniania.')
print('- Modele ML nadal sa wartosciowe jako uzupelnienie, poniewaz moga wskazywac nietypowe rekordy, ktore nie zostaly opisane recznie reguly.')
print('- Najlepsze praktyczne podejscie to hybryda: reguly dla znanych wzorcow ataku oraz ML jako dodatkowa warstwa wykrywania zachowan odstajacych.')

## 10. Wnioski koncowe do sprawozdania

1. Rzeczywiste logi auth.log zawieraja wiele typow zdarzen: udane logowania SSH, nieudane logowania SSH, proby na nieistniejacych uzytkownikow, zdarzenia PAM, sudo, sesje systemowe i CRON.
2. Najbardziej jednoznaczne anomalie w tym zbiorze to `Failed password`, `Invalid user`, `authentication failure` oraz serie nieudanych prob z jednego adresu IP.
3. Detekcja regulowa uzyskuje najlepsze wyniki, poniewaz problem zawiera wiele jawnych sygnatur bezpieczenstwa zapisanych wprost w komunikatach logow.
4. Isolation Forest i LOF maja nizszy recall, bo modele ML operuja na cechach liczbowych, a nie na pelnym znaczeniu tekstu komunikatu.
5. ML nie zastapil regul w tym przypadku, ale moze byc uzyteczny jako druga warstwa: znajduje rekordy odstajace, ktore nie musza byc objete recznie przygotowanymi regulami.
6. Do systemu produkcyjnego najlepsze byloby podejscie hybrydowe: reguly dla znanych atakow oraz ML do wykrywania nietypowych wzorcow aktywnosci.